In [25]:
from simopt.solvers.astrodf import ASTRODF
from simopt.experiment_base import ProblemSolver

'''Run is a function defined in class ProblemSolver (check simopt--> experiment--> single.py). 
It runs the solver on the problem for a given number of macroreplications (n_macroreps)
'''

'''post_replicate runs post replications at solver's recommended solutions.
Also a function defined in class ProblemSolver.
CRN is also added here. '''

exp = ProblemSolver(solver_name="ASTRODF", problem_name="CNTNEWS-1")
exp.run(n_macroreps=2)
exp.post_replicate(n_postreps=50)
print("done")

print(type(exp.all_recommended_xs))
# print(exp.all_recommended_xs)
# print(exp.all_intermediate_budgets)

xs = getattr(exp, "all_recommended_xs")
budgets = getattr(exp, "all_intermediate_budgets")

"""Which solution was recommneded at which budget. Initialised in single.py"""
for m, (x_list, b_list) in enumerate(zip(xs, budgets)):
    print(f"macrorep {m}")
    for b, x in zip(b_list, x_list):
        print("  budget = ", b, ", x = ", x)
        
"""Eg: budget 1000 means solver spent its full budget of 1000 simulation replications, 
and its final incumbent x (solution) in macrorep 1 is x = 0.22473549788741493, 
in macrorep 0 is x = 0.20995000270146888. Each is a noisy estimate."""


done
<class 'list'>
macrorep 0
  budget =  5 , x =  (0,)
  budget =  37 , x =  (0.3125,)
  budget =  43 , x =  (0.1875,)
  budget =  49 , x =  (0.22916666666666666,)
  budget =  55 , x =  (0.20833333333333334,)
  budget =  130 , x =  (0.21024068196614584,)
  budget =  297 , x =  (0.21024067741867233,)
  budget =  306 , x =  (0.21024066604998856,)
  budget =  315 , x =  (0.21024063762827913,)
  budget =  324 , x =  (0.21024056657400556,)
  budget =  333 , x =  (0.21024038893832162,)
  budget =  342 , x =  (0.21023994484911177,)
  budget =  351 , x =  (0.21023883462608714,)
  budget =  360 , x =  (0.21023605906852558,)
  budget =  369 , x =  (0.21022912017462167,)
  budget =  378 , x =  (0.2102117729398619,)
  budget =  387 , x =  (0.21016840485296248,)
  budget =  396 , x =  (0.21005998463571393,)
  budget =  414 , x =  (0.20992445936415324,)
  budget =  441 , x =  (0.20995834068204342,)
  budget =  468 , x =  (0.20994987035257087,)
  budget =  513 , x =  (0.20995119384155095,)
  budget

'Eg: budget 1000 means solver spent its full budget of 1000 simulation replications, \nand its final incumbent x (solution) in macrorep 1 is x = 0.22473549788741493, \nin macrorep 0 is x = 0.20995000270146888. Each is a noisy estimate.'

In [28]:
import numpy as np

for m in range(exp.n_macroreps):
    x = exp.all_recommended_xs[m][-1]
    reps = np.asarray(exp.all_post_replicates[m][-1]).ravel()
    est = exp.all_est_objectives[m][-1]
    print(f"macrorep {m}: x={x} f_hat={est} n={len(reps)}")
    
    

macrorep 0: x=(0.20995000270146888,) f_hat=0.2611669456216214 n=50
macrorep 1: x=(0.22473549788741493,) f_hat=0.4032957296073143 n=50


Above codes extract solutions of CNTNEWS-1. 

Instead use the Quadnoiseproblem defined previously in "cf_trial_2_example.ipynb" to extract results. 

In [31]:
from __future__ import annotations

from typing import Annotated, ClassVar

import numpy as np
from pydantic import BaseModel, Field

from mrg32k3a.mrg32k3a import MRG32k3a
from simopt import dsl
from simopt.base import (
    ConstraintType,
    Model,
    Problem,
    VariableType,
)
from simopt.input_models import Normal
# from simopt.simulations.example import (
#     Example2ModelConfig,
#     ExampleModelConfig,
#     replicate,
#     replicate_discrete,
# )
from simopt.utils import override

# Like in simulation 
class QuadNoiseConfig(BaseModel):
    x: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="decision variable"),
    ]
    noise_std: Annotated[
        float,
        Field(default=1e-8, ge=0.0, description="tiny noise standard deviation"),
    ]

class QuadNoiseModel(Model[QuadNoiseConfig]):
    """x^2 + 1 evaluated with tiny stochastic noise."""

    class_name_abbr: ClassVar[str] = "QUADNOISE"
    class_name: ClassVar[str] = "x^2 + 1 with tiny noise"
    config_class: ClassVar[type[QuadNoiseConfig]] = QuadNoiseConfig
    n_rngs: ClassVar[int] = 1
    n_responses: ClassVar[int] = 1
    
    def __init__(self, fixed_factors: dict | None = None) -> None:
        super().__init__(fixed_factors)
        self.noise_model = Normal()
        
    def replicate(self, factors: QuadNoiseConfig, rngs: list[MRG32k3a]) -> tuple[dict, dict]:
        x = np.asarray(factors.x, dtype=float)
        f = float(np.sum(x**2) + 1.0)

        # Tiny noise: set noise_std = 0.0 for exact deterministic behavior.
        noise = self.noise_model.random(rngs[0], mu=0.0, sigma=factors.noise_std)
        noisy_value = f + noise

        responses = {"est_f(x)": noisy_value}
        gradients = {"est_f(x)": {"x": 2.0 * x}}
        return responses, gradients
    
    #Similar to example.py in models.   
class QuadNoiseProblemConfig(BaseModel):
    initial_solution: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="initial point"),
    ]
    budget: Annotated[
        int,
        Field(default=1000, gt=0, description="max simulation budget"),
    ]
    
class QuadNoiseProblem(Problem):
    """Minimize x^2 + 1 under noisy stochastic replications."""

    class_name_abbr: ClassVar[str] = "QUADNOISE"
    class_name: ClassVar[str] = "Minimize x^2 + 1 with zero noise"
    config_class: ClassVar[type[BaseModel]] = QuadNoiseProblemConfig
    model_class: ClassVar[type[Model]] = QuadNoiseModel
    n_objectives: ClassVar[int] = 1
    n_stochastic_constraints: ClassVar[int] = 0
    minmax: ClassVar[tuple[int, ...]] = (-1,)
    constraint_type: ClassVar[ConstraintType] = ConstraintType.UNCONSTRAINED
    variable_type: ClassVar[VariableType] = VariableType.CONTINUOUS
    gradient_available: ClassVar[bool] = True
    model_default_factors: ClassVar[dict] = {}
    model_decision_factors: ClassVar[set[str]] = {"x"}
    
    @property
    def optimal_value(self) -> float | None:
        return 1.0
    
    @property
    def optimal_solution(self) -> tuple | None:
        return (0.0,)
    
    @override
    def build(self) -> dsl.Model:
        problem = dsl.Model()
        x0 = self.factors["initial_solution"]
        x = problem.add_continuous_vector(
            lb=-10.0, ub=10.0, shape=(len(x0),), initial=x0,
        )
        simulation = self.add_simulation(problem, {"x": x})
        problem.minimize(dsl.mean(simulation.metric("est_f(x)")))
        return problem
    
    def vector_to_factor_dict(self, vector: tuple) -> dict:
        return {"x": vector[:]}
    
    def get_random_solution(self, rand_sol_rng: MRG32k3a) -> tuple:
        # unchanged from the example
        return tuple(
            rand_sol_rng.mvnormalvariate(
                mean_vec=[0.0] * self.dim,
                cov=np.eye(self.dim).tolist(),
                factorized=False,
            )
        )

In [ ]:
# Note: For this to run above QuadNoise code should be defined. Or imported. 

import numpy as np
import pandas as pd
from simopt.solvers.astrodf import ASTRODF
from simopt.experiment_base import ProblemSolver

LOG = []
STATE = {"mrep": -1}  # to identify which macroreplication.


# Defining a class that inherits everything from astrodf.

class LoggingASTRODF(ASTRODF):

    def _initialize_solving(self, ctx):
        super()._initialize_solving(ctx)
        STATE["mrep"] += 1

    def construct_model(self, ctx):
        delta_in = self.delta_k
        out = super().construct_model(ctx)
        fval, y_var, q, grad, hessian, solns = out

        m = 2 * self.problem.dim + 1
        for i, s in enumerate(solns[-m:]):
            LOG.append({
                "mrep": STATE["mrep"],
                "iter": self.iteration_count,
                "delta_in": delta_in,
                "incumbent_x": tuple(np.ravel(self.incumbent_x)),
                "point_idx": i,
                "x": tuple(np.ravel(s.x)),
                "F_hat": float(np.ravel(fval[i])[0]),
                "sample_size": int(s.n_reps),
                "var_F": float(np.ravel(s.objectives_var)[0]),
                "model_grad": tuple(np.ravel(grad)),
                "model_hessian_diag": tuple(np.ravel(hessian)),
            })
        return out


solver = LoggingASTRODF(fixed_factors={"use_gradients": False})
problem = QuadNoiseProblem(fixed_factors={"initial_solution": (1.0,), "budget": 1000})

exp = ProblemSolver(solver=solver, problem=problem)
exp.run(n_macroreps=1, n_jobs=1)

df = pd.DataFrame(LOG)
pd.set_option("display.width", 250, "display.max_rows", None, "display.max_columns", None)
print(df[["mrep", "iter", "delta_in", "point_idx", "x", "F_hat", "sample_size"]])
df.to_csv("astrodf_design_sets_quad.csv", index=False)

     mrep  iter      delta_in  point_idx                           x       F_hat  sample_size
0       0     1  1.000000e+01          0                      (1.0,)    2.000000            5
1       0     1  1.000000e+01          1                     (9.99,)  100.800100            5
2       0     1  1.000000e+01          2                     (-9.0,)   82.000000            5
3       0     2  5.000000e+00          0                      (1.0,)    2.000000            5
4       0     2  5.000000e+00          1     (-0.05145914347922664,)    1.002648            5
5       0     2  5.000000e+00          2                      (6.0,)   37.000000            5
6       0     3  2.500000e+00          0                      (1.0,)    2.000000            5
7       0     3  2.500000e+00          1      (-1.0000000000000013,)    2.000000            5
8       0     3  2.500000e+00          2                      (3.5,)   13.250000            5
9       0     4  1.250000e+00          0                    